In [3]:
import pandas as pd
import numpy as np
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# CARGAR DATOS

In [8]:
DB_PATH = '../data/movielens.db'

conn = sqlite3.connect(DB_PATH)

ratings = pd.read_sql("SELECT * FROM ratings", conn)
ratings.columns = ['userId', 'movieId', 'rating', 'timestamp']

movies = pd.read_sql("SELECT * FROM items", conn)
movies.rename(columns={'item_id': 'movieId'}, inplace=True)

conn.close()

# SPLIT TRAIN/TEST

In [10]:
print("\n Dividiendo datos en train/test (80/20)")

train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

print(f"Train: {len(train_data):,} ratings")
print(f"Test: {len(test_data):,} ratings")

train_data.to_csv('train_data.csv', index=False)
test_data.to_csv('test_data.csv', index=False)

print(" Guardados: train_data.csv y test_data.csv")


 Dividiendo datos en train/test (80/20)
Train: 80,000 ratings
Test: 20,000 ratings
 Guardados: train_data.csv y test_data.csv


# FUNCIONES DE MÉTRICAS

In [11]:
def precision_at_k(recommended, relevant, k=10):
    """
    Precision-K: ¿De las K recomendadas, cuántas son relevantes?
    
    Args:
        recommended: lista de items recomendados
        relevant: lista de items relevantes (que le gustan al usuario)
        k: número de recomendaciones a considerar
    
    Returns:
        Precision entre 0 y 1
    """
    recommended_k = recommended[:k]
    relevant_set = set(relevant)
    hits = len([r for r in recommended_k if r in relevant_set])
    return hits / k if k > 0 else 0.0


In [12]:
def recall_at_k(recommended, relevant, k=10):
    """
    Recall-K: ¿De todos los items relevantes, cuántos logré recomendar?
    
    Args:
        recommended: lista de items recomendados
        relevant: lista de items relevantes
        k: número de recomendaciones a considerar
    
    Returns:
        Recall entre 0 y 1
    """
    recommended_k = recommended[:k]
    relevant_set = set(relevant)
    hits = len([r for r in recommended_k if r in relevant_set])
    return hits / len(relevant) if len(relevant) > 0 else 0.0

In [13]:
def average_precision(recommended, relevant):
    """
    Average Precision: Precisión considerando el orden
    
    Args:
        recommended: lista de items recomendados (ordenada)
        relevant: lista de items relevantes
    
    Returns:
        Average Precision entre 0 y 1
    """
    relevant_set = set(relevant)
    precisions = []
    hits = 0
    
    for i, item in enumerate(recommended):
        if item in relevant_set:
            hits += 1
            precisions.append(hits / (i + 1))
    
    return np.mean(precisions) if precisions else 0.0

In [14]:
def map_at_k(all_recommended, all_relevant, k=10):
    """
    MAP-K: Mean Average Precision
    Promedio de AP para todos los usuarios
    
    Args:
        all_recommended: lista de listas (recomendaciones por usuario)
        all_relevant: lista de listas (items relevantes por usuario)
        k: número de recomendaciones a considerar
    
    Returns:
        MAP entre 0 y 1
    """
    aps = []
    for recommended, relevant in zip(all_recommended, all_relevant):
        recommended_k = recommended[:k]
        ap = average_precision(recommended_k, relevant)
        aps.append(ap)
    return np.mean(aps) if aps else 0.0

In [15]:
def ndcg_at_k(recommended, relevant, k=10):
    """
    NDCG-K: Normalized Discounted Cumulative Gain
    Considera el orden ideal de las recomendaciones
    
    Args:
        recommended: lista de items recomendados
        relevant: lista de items relevantes
        k: número de recomendaciones a considerar
    
    Returns:
        NDCG entre 0 y 1
    """
    recommended_k = recommended[:k]
    relevant_set = set(relevant)
    
    # DCG
    dcg = 0.0
    for i, item in enumerate(recommended_k):
        if item in relevant_set:
            dcg += 1.0 / np.log2(i + 2)
    
    # IDCG (Ideal DCG)
    ideal_recommended = relevant[:k]
    idcg = 0.0
    for i in range(len(ideal_recommended)):
        idcg += 1.0 / np.log2(i + 2)
    
    return dcg / idcg if idcg > 0 else 0.0

In [17]:
def coverage(all_recommended, total_items):
    """
    Coverage: ¿Qué tan variadas son las recomendaciones?
    
    Args:
        all_recommended: lista de listas de recomendaciones
        total_items: número total de items disponibles
    
    Returns:
        Coverage entre 0 y 1 (porcentaje del catálogo recomendado)
    """
    unique_recommended = set()
    for recommended in all_recommended:
        unique_recommended.update(recommended)
    return len(unique_recommended) / total_items if total_items > 0 else 0.0


print(" 5 funciones de métricas definidas:")
print("   1. precision_at_k()")
print("   2. recall_at_k()")
print("   3. map_at_k()")
print("   4. ndcg_at_k()")
print("   5. coverage()")

 5 funciones de métricas definidas:
   1. precision_at_k()
   2. recall_at_k()
   3. map_at_k()
   4. ndcg_at_k()
   5. coverage()


# PREPARAR SISTEMA DE POPULARIDAD

In [18]:
# Calcular popularidad solo con train_data
movie_popularity = train_data.groupby('movieId').agg({
    'rating': ['mean', 'count']
}).reset_index()

movie_popularity.columns = ['movieId', 'avg_rating', 'num_ratings']

# Filtrar películas con al menos 5 ratings
movie_popularity = movie_popularity[movie_popularity['num_ratings'] >= 5]

# Ordenar por rating promedio
movie_popularity = movie_popularity.sort_values('avg_rating', ascending=False)

# Top 100 más populares
top_popular = movie_popularity['movieId'].head(100).tolist()

print(f" Top 100 películas populares identificadas")
print(f"   Ejemplo: {top_popular[:5]}")

 Top 100 películas populares identificadas
   Ejemplo: [1449, 114, 318, 320, 64]


# EVALUAR SISTEMA DE POPULARIDAD

In [19]:
# Tomar una muestra de usuarios para evaluar (primeros 100)
test_users = test_data['userId'].unique()[:100]

all_recommended_pop = []
all_relevant_pop = []

for user in test_users:
    # Películas que le gustaron al usuario en test (rating >= 4)
    relevant = test_data[
        (test_data['userId'] == user) & 
        (test_data['rating'] >= 4)
    ]['movieId'].tolist()
    
    if len(relevant) > 0:
        # El sistema recomienda siempre las top populares
        all_recommended_pop.append(top_popular)
        all_relevant_pop.append(relevant)

print(f" Evaluados {len(all_recommended_pop)} usuarios")

 Evaluados 100 usuarios


# CALCULAR MÉTRICAS

In [21]:
# Precision@10
precision_scores = [
    precision_at_k(rec, rel, 10) 
    for rec, rel in zip(all_recommended_pop, all_relevant_pop)
]
precision_pop = np.mean(precision_scores)

# Recall@10
recall_scores = [
    recall_at_k(rec, rel, 10) 
    for rec, rel in zip(all_recommended_pop, all_relevant_pop)
]
recall_pop = np.mean(recall_scores)

# MAP@10
map_pop = map_at_k(all_recommended_pop, all_relevant_pop, 10)

# NDCG@10
ndcg_scores = [
    ndcg_at_k(rec, rel, 10) 
    for rec, rel in zip(all_recommended_pop, all_relevant_pop)
]
ndcg_pop = np.mean(ndcg_scores)

# Coverage
total_movies = ratings['movieId'].nunique()
coverage_pop = coverage(all_recommended_pop, total_movies)

print(" Métricas calculadas")

 Métricas calculadas


In [25]:
print(" RESULTADOS - SISTEMA DE POPULARIDAD")
print("="*70)
print(f"\nPrecision-10:  {precision_pop:.4f}  ({'' if precision_pop >= 0.60 else ''} Objetivo: >0.60)")
print(f"Recall-10:     {recall_pop:.4f}  ({'' if recall_pop >= 0.40 else ''} Objetivo: >0.40)")
print(f"MAP-10:        {map_pop:.4f}")
print(f"NDCG-10:       {ndcg_pop:.4f}")
print(f"Coverage:      {coverage_pop:.4f}  ({'' if coverage_pop >= 0.30 else ''} Objetivo: >0.30)")


 RESULTADOS - SISTEMA DE POPULARIDAD

Precision-10:  0.0570  ( Objetivo: >0.60)
Recall-10:     0.0225  ( Objetivo: >0.40)
MAP-10:        0.0898
NDCG-10:       0.0484
Coverage:      0.0595  ( Objetivo: >0.30)


GUARDAR

In [28]:
resultados_popularidad = {
    'sistema': 'Popularidad',
    'precision_10': precision_pop,
    'recall_10': recall_pop,
    'map_10': map_pop,
    'ndcg_10': ndcg_pop,
    'coverage': coverage_pop
}

# Guardar como CSV
resultados_df = pd.DataFrame([resultados_popularidad])
resultados_df.to_csv('metricas_popularidad.csv', index=False)

# Guardar como pickle
with open('metricas_popularidad.pkl', 'wb') as f:
    pickle.dump(resultados_popularidad, f)

print("\n Resultados guardados:")
print("   - metricas_popularidad.csv")
print("   - metricas_popularidad.pkl")


 Resultados guardados:
   - metricas_popularidad.csv
   - metricas_popularidad.pkl


# RESUMEN / INTERPRETACION:

In [32]:
print(f"""
Precision-10 = {precision_pop:.2%}
└─> De cada 10 películas recomendadas, {precision_pop*10:.1f} le gustan al usuario

Recall-10 = {recall_pop:.2%}
└─> De todas las películas que le gustan, encontramos {recall_pop*100:.1f}%

MAP-10 = {map_pop:.2%}
└─> Qué tan bien ordenadas están las recomendaciones

NDCG-10 = {ndcg_pop:.2%}
└─> Calidad del ranking (considerando el orden ideal)

Coverage = {coverage_pop:.2%}
└─> Recomendamos {coverage_pop*100:.1f}% del catálogo total
""")

if precision_pop >= 0.60 and recall_pop >= 0.40 and coverage_pop >= 0.30:
    print(" ¡EXCELENTE! Cumples todos los objetivos")
elif precision_pop >= 0.60:
    print(" Cumples el objetivo principal de Precision-10 > 0.60")
else:
    print("  Sistema de Popularidad tiene Precision baja")
    print("   (Normal, es el sistema más básico)")


Precision-10 = 5.70%
└─> De cada 10 películas recomendadas, 0.6 le gustan al usuario

Recall-10 = 2.25%
└─> De todas las películas que le gustan, encontramos 2.3%

MAP-10 = 8.98%
└─> Qué tan bien ordenadas están las recomendaciones

NDCG-10 = 4.84%
└─> Calidad del ranking (considerando el orden ideal)

Coverage = 5.95%
└─> Recomendamos 5.9% del catálogo total

  Sistema de Popularidad tiene Precision baja
   (Normal, es el sistema más básico)
